In [39]:
import sqlite3
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

In [40]:
conn = sqlite3.connect('../data/checking-logs.sqlite')
query = """
SELECT uid, timestamp
FROM checker
WHERE uid NOT LIKE 'admin_%'
    AND labname = 'project1'
    AND status = 'ready'
    AND timestamp IS NOT NULL
ORDER BY timestamp
"""
df = pd.io.sql.read_sql(query, conn)
conn.close()

In [41]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.date

daily = df.groupby(['date', 'uid']).size().reset_index(name='count')
daily = daily.sort_values(['uid', 'date'])

dates = sorted(daily['date'].unique())
users = sorted(daily['uid'].unique())

idx = pd.MultiIndex.from_product([dates, users], names=['date', 'uid'])
full = daily.set_index(['date', 'uid']).reindex(idx, fill_value=0)
full['cumulative'] = full.groupby('uid')['count'].cumsum()
full = full.reset_index()

In [42]:
initial_df = full[full['date'] == dates[0]]
initial_traces = []
for user in users:
    row = initial_df[initial_df['uid'] == user]
    y_val = int(row['cumulative'].values[0])
    initial_traces.append(go.Scatter(
        x=[0], y=[y_val],
        mode='lines+markers', name=user,
        line=dict(width=2), marker=dict(size=4)
    ))

In [43]:
frames = []
for i, d in enumerate(dates[1:], start=1):
    frame_data = full[full['date'] <= d]
    traces = []
    for user in users:
        user_data = frame_data[frame_data['uid'] == user].sort_values('date')
        x = list(range(0, i + 1))
        y = [int(v) for v in user_data['cumulative'].values]
        traces.append(go.Scatter(
            x=x, y=y,
            mode='lines+markers', name=user,
            line=dict(width=2), marker=dict(size=4)
        ))
    frames.append(go.Frame(
        data=traces,
        name=str(d)
    ))

In [47]:
pio.templates.default = "plotly"

fig = go.Figure(
    data=initial_traces,
    layout=dict(
        width=1100,
        height=600,
        title='Dynamic of commits per user in project1',
        xaxis=dict(
            range=[0, len(dates) + 2],
            showgrid=True,
            dtick=2
        ),
        yaxis=dict(
            range=[0, int(full['cumulative'].max()) + 20],
            showgrid=True,
            dtick=20
        ),
        updatemenus=[dict(
            type='buttons',
            buttons=[dict(
                label='play',
                method='animate',
                args=[None, {'frame': {'duration': 500, 'redraw': True},
                             'fromcurrent': True}]
            )]
        )]
    ),
    frames=frames
)
fig.show()